In [ ]:
# 🚀 Correct installation order for compatibility:
!pip install --upgrade huggingface_hub==0.20.3
!pip install --upgrade diffusers==0.24.0 transformers==4.35.2 accelerate==0.25.0

: 

In [7]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch==2.0.1 torchvision==0.15.2 --extra-index-url https://download.pytorch.org/whl/cu118
!pip install xformers==0.0.21

Found existing installation: torch 2.7.1
Uninstalling torch-2.7.1:
  Successfully uninstalled torch-2.7.1


You can safely remove it manually.


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
  Using cached https://download.pytorch.org/whl/cu118/torch-2.0.1%2Bcu118-cp310-cp310-win_amd64.whl (2619.1 MB)
  Using cached https://download.pytorch.org/whl/cu118/torchvision-0.15.2%2Bcu118-cp310-cp310-win_amd64.whl (4.9 MB)

   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ------------

In [ ]:
!pip uninstall numpy -y
!pip install numpy==1.24.4


  Using cached numpy-1.24.4-cp310-cp310-win_amd64.whl.metadata (5.6 kB)
Using cached numpy-1.24.4-cp310-cp310-win_amd64.whl (14.8 MB)


In [ ]:
!mkdir -p models/streetcontrolnet       # Create the directory
%cd models/streetcontrolnet             # Move into that directory
!git clone https://huggingface.co/MisraSerenayy/controlnet-topo-street-lora-1.1
%cd ../..                               # Go back to project root (optional, only if you need it)


In [ ]:
!huggingface-cli login --token YOUR_TOKEN_HERE
 


Token will not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\DELL\.cache\huggingface\token
Login successful


In [3]:

import numpy



The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


ImportError: cannot import name 'core' from partially initialized module 'numpy' (most likely due to a circular import) (c:\Users\DELL\anaconda3\envs\streetenv\lib\site-packages\numpy\__init__.py)

In [15]:
# 4️⃣ Download base ControlNet model
from diffusers import ControlNetModel
controlnet_base = ControlNetModel.from_pretrained("lllyasviel/control_v11f1p_sd15_depth")
controlnet_base.save_pretrained("base_controlnet_config")

In [16]:
import shutil
import os

# Paths (edit if needed)
src_config = "base_controlnet_config/config.json"
dst_config = "models/streetcontrolnet/controlnet-topo-street-lora-1.1/checkpoint-10000/controlnet/config.json"
model_path = "models/streetcontrolnet/controlnet-topo-street-lora-1.1/checkpoint-10000/controlnet/model.safetensors"
new_model_path = "models/streetcontrolnet/controlnet-topo-street-lora-1.1/checkpoint-10000/controlnet/diffusion_pytorch_model.safetensors"

# Copy config.json
shutil.copy(src_config, dst_config)

# Rename model.safetensors
if os.path.exists(model_path):
    os.rename(model_path, new_model_path)
else:
    print("model.safetensors not found!")


model.safetensors not found!


In [38]:

# 10️⃣ Load control image
# Dummy blank image as init image to test:
init_image = Image.open("generated topo.png").convert("RGB").resize((512, 512))
control_image = Image.open("generated topo.png").convert("RGB").resize((512, 512))


In [ ]:
import torch
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline
from PIL import Image

# --- Paths ---
controlnet_model_path = "models/streetcontrolnet/controlnet-topo-street-lora-1.1/checkpoint-10000/controlnet"
base_model_path = "runwayml/stable-diffusion-v1-5"   # leave as-is if using HuggingFace hub           
output_path = "./inference_output.png"


# --- Load models ---
controlnet = ControlNetModel.from_pretrained(controlnet_model_path, torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    base_model_path,
    controlnet=controlnet,
    torch_dtype=torch.float16
)
pipe.to("cuda")
pipe.text_encoder.to(dtype=torch.float16)

# --- Prompts ---
positive_prompt = "dense detailed street network , map, schematic, high contrast, only black and white, sharp"
negative_prompt = ("no satellite, no text, no buildings, no cars, no water, no trees, no blurry, no labels, no numbers, no thick lines, no map key, no border")

# --- Inference ---
guidance_scale = 15.0
num_inference_steps = 150
seed = 0
generator = torch.manual_seed(seed)

output = pipe(
    prompt=positive_prompt,
    negative_prompt=negative_prompt,
    image=init_image,
    control_image=control_image,
    guidance_scale=guidance_scale,
    num_inference_steps=num_inference_steps,
    generator=generator,
    controlnet_conditioning_scale=1.0,
    strength=0.3,
)
result = output.images[0]
result.save(output_path)
result.show()
print(f"Inference done. Output saved as {output_path}.")


Loading pipeline components...:  71%|███████▏  | 5/7 [00:03<00:01,  1.49it/s]`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["bos_token_id"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["eos_token_id"]` will be overriden.
100%|██████████| 150/150 [00:27<00:00,  5.45it/s]


Inference done. Output saved as ./inference_output.png.
